In [2]:
import pandas as pd
import os
import hashlib
import numpy as np
from pgvector.psycopg import register_vector
import psycopg
from PIL import Image
import torch
from torchvision import models, transforms

In [ ]:
# Path to the CSV file
CSV_FILE = "/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/_tinch_picture_overview_1k__202411250857withCLIENT_DBID.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(CSV_FILE)

df['PICTURE_PATH_AT_HOME'][:1]
# df['CLIENT_DBID'].nunique() #775

# df.columns
df.head()

,CLIENT_DBID,PICTURE_1,URL_PICTURE,PICTURE_PATH,CI_MATRIXIMAGEID,VENCATNUM,VENDOR_NAME,VENDOR_NAME_PARENT,MFRCATNUM,MANUFACTURER_NAME,MANUFACTURER_NAME_PARENT,UNIQUE_ID,PICTURE_PATH_AT_HOME
0,SLC_202005_004035,THERMOMETER_DIGITAL_150779E_FISHERHEALTHCAREIN...,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,"b88f1cf4-0f0a-6ef2-d356-494cc118e25f,7e9018f5-...",15-077-9E,FISHER HEALTHCARE INC,THERMO FISHER SCIENTIFIC INC,15-077-9E,FISHER SCIENTIFIC COMPANY LLC,THERMO FISHER SCIENTIFIC INC,6568ef08-01df-4a30-a5a9-8b335b262648,/home/tinchung/Documents/Bestarion_Intern/OneD...
1,SLC_202005_004049,PAD_FLOOR--MACHINE_MMM7200_PRI01.JPG,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,7174d33c-2c79-b18d-3cf5-57910b192f49,1001712,NETWORK SERVICES COMPANY,NETWORK SERVICES COMPANY,7200-20IN,3M COMPANY,3M COMPANY,c06cc9ab-2095-4a5e-8fe9-4dea8d6637b4,/home/tinchung/Documents/Bestarion_Intern/OneD...
2,UHS_202005_004267,DISPLAY_DOCUMENT_30305_DocumentDisplay1HEP3_AS...,http://www.grainger.com/Grainger/TARIFOLD-Docu...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,78b27d0c-0175-b36c-d182-531779f95d09,1HEP3,WW GRAINGER INC,WW GRAINGER INC,DA271,TARIFOLD INC,TARIFOLD INC,a0f29b64-1b78-49fa-a2ee-f928f993f945,/home/tinchung/Documents/Bestarion_Intern/OneD...
3,UHS_202005_004537,KIT_SEROLOGY--TEST_TEST_ROM5025_CLINICALINNOVA...,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,24897150-0dbf-2545-2188-0d5a1723fae4,ROM-5025,CARDINAL HEALTH INC,CARDINAL HEALTH INC,ROM-5025,LABORIE MEDICAL TECHNOLOGIES CORPORATION,WALLENBERG FOUNDATIONS AB,2e7850af-80cf-48c8-933e-0bd696850f36,/home/tinchung/Documents/Bestarion_Intern/OneD...
4,UHS_202005_004730,OXIMETER_FINGERTIP_MMO9709_PRI01.JPG,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,7f0017cd-8bc6-dfd6-89d9-b6379996983c,9909,MASIMO CORPORATION,MASIMO CORPORATION,9909,MASIMO CORPORATION,MASIMO CORPORATION,e5527a75-f44b-4068-9916-13b9029cfc16,/home/tinchung/Documents/Bestarion_Intern/OneD...


In [6]:
import pandas as pd

# Path to the CSV file
CSV_FILE = "/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/_tinch_picture_overview_1k__202411250857withCLIENT_DBID.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(CSV_FILE)

# Define the base path
BASE_PATH = '/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/'

# Create the new column with the concatenated path
df['PICTURE_PATH_AT_HOME'] = BASE_PATH + df['PICTURE_1']

# Save the updated DataFrame back to the CSV file
df.to_csv(CSV_FILE, index=False)
		

In [ ]:
# Database configuration
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = "nhui6112002"
DB_HOST = "localhost"
DB_PORT = "5400"

# Path to the CSV file
CSV_FILE = "/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/_tinch_picture_overview_1k__202411250857withCLIENT_DBID.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(CSV_FILE)

# Define the base path
BASE_PATH = '/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/'

# Create the new column with the concatenated path
df['PICTURE_PATH_AT_HOME'] = BASE_PATH + df['PICTURE_1']

# Function to insert data into the PostgreSQL table
def insert_data_to_db(df):
    conn = psycopg2.connect(
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
    )
    cursor = conn.cursor()
    
    # Insert data row by row
    for index, row in df.iterrows():
        cursor.execute(
            """
            INSERT INTO tinch_picture_overview_1k_20241125 (
                "CLIENT_DBID", "PICTURE_1", "URL_PICTURE", "PICTURE_PATH", 
                "PICTURE_PATH_AT_HOME", "CI_MATRIXIMAGEID", "VENCATNUM", 
                "VENDOR_NAME", "VENDOR_NAME_PARENT", "MFRCATNUM", 
                "MANUFACTURER_NAME", "MANUFACTURER_NAME_PARENT"
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """,
            (
                row['CLIENT_DBID'],
                row['PICTURE_1'],
                row['URL_PICTURE'],
                row['PICTURE_PATH'],
                row['PICTURE_PATH_AT_HOME'],
                row['CI_MATRIXIMAGEID'],
                row['VENCATNUM'],
                row['VENDOR_NAME'],
                row['VENDOR_NAME_PARENT'],
                row['MFRCATNUM'],
                row['MANUFACTURER_NAME'],
                row['MANUFACTURER_NAME_PARENT']
            )
        )
    
    conn.commit()
    cursor.close()
    conn.close()

if __name__ == "__main__":
    insert_data_to_db(df)

In [13]:
df.head()

,PICTURE_1,URL_PICTURE,PICTURE_PATH,PICTURE_PATH_AT_HOME,CI_MATRIXIMAGEID,VENCATNUM,VENDOR_NAME,VENDOR_NAME_PARENT,MFRCATNUM,MANUFACTURER_NAME,MANUFACTURER_NAME_PARENT,UNIQUE_ID
0,MOUTHPIECE_RESPIRATORY_1055598_CLS.jpg,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,/home/tinchung/Documents/Bestarion_Intern/OneD...,f5b8aad8-ae23-36fb-c9b7-aef745877f4a,1055598,MCKESSON MEDICAL-SURGICAL INC,MCKESSON CORPORATION,141-5050-50,MCKESSON MEDICAL-SURGICAL INC,MCKESSON CORPORATION,58a21e98-bbcc-4e04-a932-1704de95ee83
1,COVER_PROTECTIVE_SWDPMAC10NCC_PRI01.1.JPG,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,/home/tinchung/Documents/Bestarion_Intern/OneD...,195aa918-7386-b703-c8f0-f62c3ecff385,SWDPMAC10NCC,MEDLINE INDUSTRIES INC,MEDLINE INDUSTRIES INC,PMAC10N-CC,MEDTRONIC INC,MEDTRONIC PLC,1d1c035b-cdb2-4586-b9f2-1f0cb28c340e
2,DRESSING_FOAM_MSC1200B_MEDLINEINDUSTRIESINC.jpg,NaN,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,/home/tinchung/Documents/Bestarion_Intern/OneD...,29d9399a-57d0-6a30-7258-e70f96f1cf8c,MSC1200B,MEDLINE INDUSTRIES INC,MEDLINE INDUSTRIES INC,MSC1200B,MEDLINE INDUSTRIES INC,MEDLINE INDUSTRIES INC,23678554-7254-4f3c-9c66-81c9c4a84cd1
3,SCREW_CANNULATED--INTERFERENCE_AR18720V08_ARTH...,NaN,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,/home/tinchung/Documents/Bestarion_Intern/OneD...,a56bd71a-573d-a8e0-ed03-a500d5231f01,AR-18720V-13,ARTHREX INC,ARTHREX INC,AR-18720V-13,ARTHREX INC,ARTHREX INC,472506df-6e80-41a6-9180-ba225bcca3e1
4,BOARD_CUTTING_MDS4662242_PRI01.JPG,https://s3.amazonaws.com/msss-assets/productio...,https://svn.elarion.com/Repository/SDC/SDD1/Pr...,/home/tinchung/Documents/Bestarion_Intern/OneD...,"ab95cd99-6daf-01d7-a6be-f1b688414685,82c2336f-...",MDS4662242,MEDLINE INDUSTRIES INC,MEDLINE INDUSTRIES INC,MDS4662242,MEDLINE INDUSTRIES INC,MEDLINE INDUSTRIES INC,9c1795be-7aad-4693-b94b-e890241e8398


In [ ]:
# Load pre-trained ResNet model
model = models.resnet50(pretrained=True)
model.eval()  # Set to evaluation mode

# Remove the final classification layer to get embeddings
model = torch.nn.Sequential(*list(model.children())[:-1])

# Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def get_image_embedding(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = preprocess(image).unsqueeze(0)
        with torch.no_grad():
            embedding = model(image_tensor)
        return embedding.squeeze().numpy()
    except (OSError, IOError) as e:
        print(f"Error processing {image_path}: {e}")
        return None

def save_embedding_to_db(image_path, embedding, unique_id):
    conn = psycopg2.connect(
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
    )
    cursor = conn.cursor()
    cursor.execute(
        """
        UPDATE tinch_picture_overview_1k_202411211110
        SET embedding = %s
        WHERE "UNIQUE_ID" = %s
        """,
        (embedding.tolist(), unique_id)
    )
    conn.commit()
    cursor.close()
    conn.close()

def process_images(image_folder):
    for image_name in os.listdir(image_folder):
        image_path = os.path.join(image_folder, image_name)
        if os.path.isfile(image_path) and image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            embedding = get_image_embedding(image_path)
            if embedding is not None:
                # Assuming the UNIQUE_ID is derived from the image name or another unique identifier
                unique_id = image_name.split('.')[0]  # Adjust this as needed
                save_embedding_to_db(image_path, embedding, unique_id)

if __name__ == "__main__":
    process_images(IMAGE_FOLDER)

Error processing /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/BANDAGE_TUBULAR-SUPPORT_RETAINER_DRESSING_4344.jpg: image file is truncated (6 bytes not processed)


## Generating and Storing Embeddings:

In [15]:
# Load pre-trained ResNet model
model = models.resnet50(pretrained=True)
model.eval()  # Set to evaluation mode

# Remove the final classification layer to get embeddings
model = torch.nn.Sequential(*list(model.children())[:-1])

# Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/clip-env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/clip-env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [16]:
def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        embedding = model(image_tensor)
    return embedding.squeeze().numpy()

def save_embedding_to_db(row, embedding):
    conn = psycopg2.connect(
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
    )
    cursor = conn.cursor()
    cursor.execute(
        """
        INSERT INTO tinch_picture_overview_1k_20241125 (
            "CLIENT_DBID", "PICTURE_1", "URL_PICTURE", "PICTURE_PATH", 
            "PICTURE_PATH_AT_HOME", "CI_MATRIXIMAGEID", "VENCATNUM", 
            "VENDOR_NAME", "VENDOR_NAME_PARENT", "MFRCATNUM", 
            "MANUFACTURER_NAME", "MANUFACTURER_NAME_PARENT", "embedding"
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (
            row['CLIENT_DBID'],
            row['PICTURE_1'],
            row['URL_PICTURE'],
            row['PICTURE_PATH'],
            row['PICTURE_PATH_AT_HOME'],
            row['CI_MATRIXIMAGEID'],
            row['VENCATNUM'],
            row['VENDOR_NAME'],
            row['VENDOR_NAME_PARENT'],
            row['MFRCATNUM'],
            row['MANUFACTURER_NAME'],
            row['MANUFACTURER_NAME_PARENT'],
            embedding.tolist()
        )
    )
    conn.commit()
    cursor.close()
    conn.close()

def process_images():
    # Read the CSV file into a DataFrame
    df = pd.read_csv(CSV_FILE)

    # Define the base path
    BASE_PATH = '/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/'

    # Create the new column with the concatenated path
    df['PICTURE_PATH_AT_HOME'] = BASE_PATH + df['PICTURE_1']

    for index, row in df.iterrows():
        image_path = row['PICTURE_PATH_AT_HOME']
        if os.path.isfile(image_path) and row['PICTURE_1'].lower().endswith(('.png', '.jpg', '.jpeg')):
            embedding = get_image_embedding(image_path)
            save_embedding_to_db(row, embedding)

if __name__ == "__main__":
    process_images()

UniqueViolation: duplicate key value violates unique constraint "tinch_picture_overview_1k_20241125_pkey"
DETAIL:  Key ("CLIENT_DBID")=(SLC_202005_004035) already exists.


In [7]:


# Database configuration
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = "nhui6112002"
DB_HOST = "localhost"
DB_PORT = "5400"

# Folder containing images
IMAGE_FOLDER = "/home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture"

# Load pre-trained ResNet model
model = models.resnet50(pretrained=True)
model.eval()  # Set to evaluation mode

# Remove the final classification layer to get embeddings
model = torch.nn.Sequential(*list(model.children())[:-1])

# Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        embedding = model(image_tensor)
    embedding = embedding.squeeze()[:2000].numpy()  # Truncate to 2000 dimensions
    return embedding

def compute_embedding_hash(embedding):
    embedding_bytes = embedding.tobytes()
    return hashlib.md5(embedding_bytes).digest()

def main():
    # Collect image paths, embeddings, and hashes
    image_paths = []
    embeddings_list = []
    embedding_hashes = set()

    unique_embeddings = []
    unique_embedding_hashes = []

    for image_name in os.listdir(IMAGE_FOLDER):
        image_path = os.path.join(IMAGE_FOLDER, image_name)
        if os.path.isfile(image_path) and image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                embedding = get_image_embedding(image_path)
                embedding_hash = compute_embedding_hash(embedding)
                if embedding_hash not in embedding_hashes:
                    embedding_hashes.add(embedding_hash)
                    unique_embeddings.append(embedding.tolist())
                    unique_embedding_hashes.append(embedding_hash)
                image_paths.append(image_path)
                embeddings_list.append(embedding.tolist())
            except Exception as e:
                print(f"Error processing {image_path}: {e}")

    # Connect to the database
    conn = psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
        autocommit=True
    )
    register_vector(conn)

    # Create tables
    with conn.cursor() as cur:
        cur.execute('''
            DROP TABLE IF EXISTS "embeddings_table" cascade;
            CREATE TABLE embeddings_table (
                id serial PRIMARY KEY,
                embedding vector(2000),
                embedding_hash bytea UNIQUE
            );
        ''')

        cur.execute('''
            DROP TABLE IF EXISTS "image_embeddings_map" cascade;
            CREATE TABLE image_embeddings_map (
                id serial PRIMARY KEY,
                image_path text,
                embedding_id integer REFERENCES embeddings_table(id)
            );
    ''')

    # Insert unique embeddings with their hashes
    print(f'Loading {len(unique_embeddings)} unique embeddings')
    with conn.cursor() as cur:
        with cur.copy('COPY embeddings_table (embedding, embedding_hash) FROM STDIN WITH (FORMAT BINARY)') as copy:
            copy.set_types(['vector', 'bytea'])
            for embedding, embedding_hash in zip(unique_embeddings, unique_embedding_hashes):
                copy.write_row([embedding, embedding_hash])

    # Map image paths to embedding IDs
    # First, create a dictionary mapping embedding hashes to their IDs
    embedding_hash_to_id = {}
    with conn.cursor() as cur:
        cur.execute('SELECT embedding_hash, id FROM embeddings_table')
        rows = cur.fetchall()
        embedding_hash_to_id = {row[0]: row[1] for row in rows}

    # Prepare data for image_embeddings_map
    image_embeddings_map = []
    for image_path, embedding in zip(image_paths, embeddings_list):
        embedding_hash = compute_embedding_hash(np.array(embedding))
        embedding_id = embedding_hash_to_id.get(embedding_hash)
        if embedding_id:
            image_embeddings_map.append((image_path, embedding_id))
        else:
            print(f"No embedding ID found for {image_path}")

    # Insert image paths and embedding IDs into image_embeddings_map
    print(f'Inserting {len(image_embeddings_map)} image paths')
    with conn.cursor() as cur:
        cur.executemany(
            'INSERT INTO image_embeddings_map (image_path, embedding_id) VALUES (%s, %s)',
            image_embeddings_map
        )

    # Create indexes
    with conn.cursor() as cur:
        cur.execute('CREATE INDEX ON embeddings_table USING hnsw (embedding vector_cosine_ops)')

    # Update planner statistics
    with conn.cursor() as cur:
        cur.execute('ANALYZE embeddings_table')
        cur.execute('ANALYZE image_embeddings_map')

    conn.close()

if __name__ == "__main__":
    main()

Error processing /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/BANDAGE_TUBULAR-SUPPORT_RETAINER_DRESSING_4344.jpg: image file is truncated (6 bytes not processed)
Loading 635 unique embeddings
No embedding ID found for /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/FORCEPS_VASCULAR_MDS1247119_HRE01.jpg
No embedding ID found for /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/GUIDE_INSTRUMENT_C8431_HORTONAUTOMATICSINC.jpg
No embedding ID found for /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/BAG_PLASTIC_S5372_ULINECORPORATION.jpg
No embedding ID found for /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/SUTURE_NON-ABSORBABLE_G242337_NOVOSURGICALINC.jpg
No embedding ID found for /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22-2024/Picture/BELT_SAFETY_TORSO_516303659L.jpg
No embedding ID found for /home/tinchung/Documents/Bestarion_Intern/OneDrive_1_11-22